In [1]:
!nvidia-smi

Sat Feb 15 22:47:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
%%bash
cat > cuda_util_kernels.cu << 'EOF'
// cuda_util_kernels.cu
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <vector>
#include <cmath>

// Tiling parameters for modulate kernel.
#define TILE_T 32
#define TILE_D 32

//-------------------------------------------------------------------------
// 1. Modulate Kernel
// Computes for each sample (n):
//   out[n, t, d] = x[n, t, d] * (1 + scale[n, d]) + shift[n, d]
// x: [N, T, D], shift/scale: [N, D]
__global__ void modulate_kernel(const float* __restrict__ x,
                                const float* __restrict__ shift,
                                const float* __restrict__ scale,
                                float* __restrict__ out,
                                int T, int D)
{
    int n = blockIdx.z;           // sample index
    int t_start = blockIdx.x * TILE_T;
    int d_start = blockIdx.y * TILE_D;
    int t_idx = threadIdx.x;
    int d_idx = threadIdx.y;
    int t = t_start + t_idx;
    int d = d_start + d_idx;

    // Load per-sample shift/scale into shared memory
    __shared__ float s_shared[TILE_D];
    __shared__ float sh_shared[TILE_D];
    for (int i = d_idx; i < TILE_D; i += blockDim.y) {
        int d_global = d_start + i;
        if (d_global < D) {
            s_shared[i] = scale[n * D + d_global];
            sh_shared[i] = shift[n * D + d_global];
        }
    }
    __syncthreads();

    if (t < T && d < D) {
        int idx = n * T * D + t * D + d;
        float s_val = s_shared[d_idx];
        float sh_val = sh_shared[d_idx];
        out[idx] = x[idx] * (1.0f + s_val) + sh_val;
    }
}

//-------------------------------------------------------------------------
// 2. SinCos Positional Embedding Kernel
// Computes a 2D sincos positional embedding for a grid of size (grid_size x grid_size).
// The output tensor has shape: [grid_size*grid_size, embed_dim].
// We split embed_dim into two halves (for y and x) and, for each half,
// compute sin and cos values using frequencies based on 10000.
__global__ void sincos_pos_embed_kernel(float* __restrict__ pos_embed,
                                          int embed_dim, int grid_size)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int num_positions = grid_size * grid_size;
    if (idx < num_positions) {
        int i = idx / grid_size;  // row (y coordinate)
        int j = idx % grid_size;  // column (x coordinate)
        int D = embed_dim;
        int half = D / 2;
        int quarter = half / 2;
        float* out_ptr = pos_embed + idx * D;
        // For the y coordinate:
        for (int k = 0; k < quarter; k++) {
            float omega = expf(-logf(10000.0f) * ((float)k / (float)half));
            float val = i * omega;
            out_ptr[k] = sinf(val);
            out_ptr[k + quarter] = cosf(val);
        }
        // For the x coordinate:
        for (int k = 0; k < quarter; k++) {
            float omega = expf(-logf(10000.0f) * ((float)k / (float)half));
            float val = j * omega;
            out_ptr[half + k] = sinf(val);
            out_ptr[half + k + quarter] = cosf(val);
        }
    }
}

//-------------------------------------------------------------------------
// 3. Wrappers to be called from Python
// These functions take PyTorch tensors as input, launch the CUDA kernels,
// and return output tensors.

// modulate_forward:
//   x: [N, T, D], shift/scale: [N, D]
//   Returns: [N, T, D]
torch::Tensor modulate_forward(torch::Tensor x, torch::Tensor shift, torch::Tensor scale) {
    TORCH_CHECK(x.is_cuda(), "x must be a CUDA tensor");
    TORCH_CHECK(shift.is_cuda(), "shift must be a CUDA tensor");
    TORCH_CHECK(scale.is_cuda(), "scale must be a CUDA tensor");

    int N = x.size(0);
    int T = x.size(1);
    int D = x.size(2);
    auto out = torch::empty_like(x);
    dim3 blockDim(TILE_T, TILE_D, 1);
    dim3 gridDim((T + TILE_T - 1) / TILE_T,
                 (D + TILE_D - 1) / TILE_D,
                 N);
    modulate_kernel<<<gridDim, blockDim>>>(
        x.data_ptr<float>(),
        shift.data_ptr<float>(),
        scale.data_ptr<float>(),
        out.data_ptr<float>(),
        T, D);
    cudaDeviceSynchronize();
    return out;
}

// sincos_pos_embed_forward:
//   Given embed_dim and grid_size, returns a tensor of shape [grid_size*grid_size, embed_dim]
torch::Tensor sincos_pos_embed_forward(int embed_dim, int grid_size) {
    int num_positions = grid_size * grid_size;
    auto options = torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCUDA);
    auto pos_embed = torch::empty({num_positions, embed_dim}, options);
    int threadsPerBlock = 256;
    int blocks = (num_positions + threadsPerBlock - 1) / threadsPerBlock;
    sincos_pos_embed_kernel<<<blocks, threadsPerBlock>>>(
        pos_embed.data_ptr<float>(), embed_dim, grid_size);
    cudaDeviceSynchronize();
    return pos_embed;
}

//-------------------------------------------------------------------------
// 4. Bindings (using pybind11)
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("modulate_forward", &modulate_forward, "Modulate forward (CUDA)");
    m.def("sincos_pos_embed_forward", &sincos_pos_embed_forward, "Sincos Pos Embed forward (CUDA)");
}

EOF


In [3]:
!pip install ninja

In [4]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [34]:
import torch
from torch.utils.cpp_extension import load

# Use the full path from your output
cuda_util = load(
    name="cuda_util_kernels",
    sources=["/content/cuda_util_kernels.cu"],  # Full path from your output
    verbose=True,
    extra_cuda_cflags=['--expt-relaxed-constexpr'],

)

Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
No modifications detected for re-loaded extension module cuda_util_kernels, skipping build step...
Loading extension module cuda_util_kernels...


In [39]:
import torch
import numpy as np

# --- Test 1: modulate_forward ---
# PyTorch reference: out = x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)
def test_modulate():
    # Create random tensors
    N, T, D = 2, 64, 128
    x = torch.rand(N, T, D, device='cuda')
    shift = torch.full((N, D), 0.1, device='cuda')
    scale = torch.full((N, D), 0.2, device='cuda')

    # PyTorch reference implementation
    expected = x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

    # Call your CUDA extension
    output = cuda_util.modulate_forward(x, shift, scale)

    # Compare (using a tolerance)
    assert torch.allclose(output, expected, atol=1e-5), "modulate_forward test failed"
    print("modulate_forward test passed.")

# --- Test 2: sincos_pos_embed_forward ---
# PyTorch reference: We'll re-implement the reference (as in models.py) in Python.
def get_2d_sincos_pos_embed(embed_dim, grid_size):
    # Create grid of positions
    grid_h = np.arange(grid_size, dtype=np.float32)
    grid_w = np.arange(grid_size, dtype=np.float32)
    grid = np.meshgrid(grid_w, grid_h)  # note: x then y
    grid = np.stack(grid, axis=0)  # shape [2, grid_size, grid_size]
    grid = grid.reshape(2, 1, grid_size, grid_size)

    # We'll compute the embedding similarly to the CUDA kernel.
    num_positions = grid_size * grid_size
    pos_embed = np.empty((num_positions, embed_dim), dtype=np.float32)
    half = embed_dim // 2
    quarter = half // 2
    for idx in range(num_positions):
        i = idx // grid_size
        j = idx % grid_size
        emb = np.empty(embed_dim, dtype=np.float32)
        for k in range(quarter):
            omega = np.exp(-np.log(10000) * (k / half))
            val = i * omega
            emb[k] = np.sin(val)
            emb[k + quarter] = np.cos(val)
        for k in range(quarter):
            omega = np.exp(-np.log(10000) * (k / half))
            val = j * omega
            emb[half + k] = np.sin(val)
            emb[half + k + quarter] = np.cos(val)
        pos_embed[idx, :] = emb
    return pos_embed

def test_sincos_pos_embed():
    grid_size = 16
    embed_dim = 128
    # Get CUDA output
    pos_embed_cuda = cuda_util.sincos_pos_embed_forward(embed_dim, grid_size)
    pos_embed_cuda_np = pos_embed_cuda.cpu().numpy()

    # PyTorch reference output
    pos_embed_ref = get_2d_sincos_pos_embed(embed_dim, grid_size)

    # Compare
    assert np.allclose(pos_embed_cuda_np, pos_embed_ref, atol=1e-5), "sincos_pos_embed_forward test failed"
    print("sincos_pos_embed_forward test passed.")

# Run tests
test_modulate()
test_sincos_pos_embed()


modulate_forward test passed.
sincos_pos_embed_forward test passed.


In [7]:
%%bash
cat > cuda_timestep_embed.cu << 'EOF'
// cuda_timestep_embed.cu
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <math.h>

//---------------------------------------------------------
// 1. Sinusoidal Timestep Embedding Kernel
//
// Inputs:
//  - t: [N] (timesteps)
//  - freqs: [half] (precomputed frequencies, where half = frequency_embedding_size/2)
// Output:
//  - embed: [N, dim] (with dim = 2 * half)
// For each sample n and each dimension d:
//    if d < half: embed[n,d] = cos(t[n] * freqs[d])
//    else:        embed[n,d] = sin(t[n] * freqs[d-half])
//
__global__ void timestep_embed_kernel(const float* __restrict__ t,
                                        const float* __restrict__ freqs,
                                        float* __restrict__ embed,
                                        int N, int dim, int half) {
    int n = blockIdx.x * blockDim.x + threadIdx.x;
    int d = blockIdx.y * blockDim.y + threadIdx.y;
    if (n < N && d < dim) {
        __shared__ float freqs_shared[256]; // Adjust if needed.
        // Each thread in y-direction cooperatively loads freqs.
        for (int i = threadIdx.y; i < half; i += blockDim.y) {
            freqs_shared[i] = freqs[i];
        }
        __syncthreads();

        float t_val = t[n];
        int index = n * dim + d;
        if (d < half) {
            embed[index] = cosf(t_val * freqs_shared[d]);
        } else {
            embed[index] = sinf(t_val * freqs_shared[d - half]);
        }
    }
}

//---------------------------------------------------------
// 2. Simple Shared-Memory Tiled GEMM for Linear Layer (no WMMA)
// Computes: out = A * B^T + bias
//  - A: [N, K] (input)
//  - B: [M, K] (weight matrix, row-major)
//  - bias: [M]
//  - out: [N, M]
// We compute the product A * B^T (so each row of A multiplies each row of B)
// using a tiled GEMM algorithm with tile size TILE_DIM x TILE_DIM.
#define TILE_DIM 16

__global__ void linear_kernel(const float* __restrict__ A,
                              const float* __restrict__ B,
                              const float* __restrict__ bias,
                              float* __restrict__ out,
                              int N, int K, int M) {
    int row = blockIdx.y * TILE_DIM + threadIdx.y;
    int col = blockIdx.x * TILE_DIM + threadIdx.x;
    float sum = 0.0f;
    __shared__ float sA[TILE_DIM][TILE_DIM];
    __shared__ float sB[TILE_DIM][TILE_DIM];

    // Loop over tiles along the K dimension.
    for (int t = 0; t < (K + TILE_DIM - 1) / TILE_DIM; t++) {
        int A_col = t * TILE_DIM + threadIdx.x;
        if (row < N && A_col < K)
            sA[threadIdx.y][threadIdx.x] = A[row * K + A_col];
        else
            sA[threadIdx.y][threadIdx.x] = 0.0f;

        int B_row = t * TILE_DIM + threadIdx.y;
        // Here, B is [M, K] in row-major order.
        // We need B^T for multiplication, so we access B[col * K + B_row].
        if (col < M && B_row < K)
            sB[threadIdx.y][threadIdx.x] = B[col * K + B_row];
        else
            sB[threadIdx.y][threadIdx.x] = 0.0f;

        __syncthreads();

        for (int i = 0; i < TILE_DIM; i++) {
            sum += sA[threadIdx.y][i] * sB[i][threadIdx.x];
        }
        __syncthreads();
    }

    if (row < N && col < M) {
        out[row * M + col] = sum + bias[col];
    }
}

//---------------------------------------------------------
// 3. Tiled GEMM with SiLU Activation for Linear+SiLU Layer
// Computes: out = SiLU( A * B^T + bias )
// where SiLU(x) = x * sigmoid(x), and sigmoid(x) = 1/(1+exp(-x)).
__global__ void linear_silu_kernel(const float* __restrict__ A,
                                   const float* __restrict__ B,
                                   const float* __restrict__ bias,
                                   float* __restrict__ out,
                                   int N, int K, int M) {
    int row = blockIdx.y * TILE_DIM + threadIdx.y;
    int col = blockIdx.x * TILE_DIM + threadIdx.x;
    float sum = 0.0f;
    __shared__ float sA[TILE_DIM][TILE_DIM];
    __shared__ float sB[TILE_DIM][TILE_DIM];

    for (int t = 0; t < (K + TILE_DIM - 1) / TILE_DIM; t++) {
        int A_col = t * TILE_DIM + threadIdx.x;
        if (row < N && A_col < K)
            sA[threadIdx.y][threadIdx.x] = A[row * K + A_col];
        else
            sA[threadIdx.y][threadIdx.x] = 0.0f;

        int B_row = t * TILE_DIM + threadIdx.y;
        if (col < M && B_row < K)
            sB[threadIdx.y][threadIdx.x] = B[col * K + B_row];
        else
            sB[threadIdx.y][threadIdx.x] = 0.0f;

        __syncthreads();

        for (int i = 0; i < TILE_DIM; i++) {
            sum += sA[threadIdx.y][i] * sB[i][threadIdx.x];
        }
        __syncthreads();
    }

    if (row < N && col < M) {
        float x = sum + bias[col];
        float sig = 1.f / (1.f + expf(-x));
        out[row * M + col] = x * sig;
    }
}

//---------------------------------------------------------
// 4. Python Wrapper Functions
//
// (a) Timestep embedding forward.
torch::Tensor timestep_embedding_forward(torch::Tensor t, torch::Tensor freqs, int dim) {
    TORCH_CHECK(t.is_cuda(), "t must be a CUDA tensor");
    TORCH_CHECK(freqs.is_cuda(), "freqs must be a CUDA tensor");
    int N = t.size(0);
    int half = dim / 2;
    auto options = torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCUDA);
    auto embed = torch::empty({N, dim}, options);
    dim3 blockDim(16, 16);
    dim3 gridDim((N + blockDim.x - 1) / blockDim.x, (dim + blockDim.y - 1) / blockDim.y);
    timestep_embed_kernel<<<gridDim, blockDim>>>(
        t.data_ptr<float>(), freqs.data_ptr<float>(), embed.data_ptr<float>(), N, dim, half);
    cudaDeviceSynchronize();
    return embed;
}

// (b) Linear layer forward (computes A * weight^T + bias).
//  - A: [N, K]
//  - weight: [M, K]  (each row is one output's weights)
//  - bias: [M]
//  Output: [N, M]
torch::Tensor linear_forward(torch::Tensor A, torch::Tensor weight, torch::Tensor bias) {
    TORCH_CHECK(A.is_cuda() && weight.is_cuda() && bias.is_cuda(), "Tensors must be CUDA");
    int N = A.size(0);
    int K = A.size(1);
    int M = weight.size(0);
    auto options = torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCUDA);
    auto out = torch::empty({N, M}, options);
    dim3 blockDim(TILE_DIM, TILE_DIM);
    // Grid dimensions: output is [N, M]
    dim3 gridDim((M + blockDim.x - 1) / blockDim.x, (N + blockDim.y - 1) / blockDim.y);
    linear_kernel<<<gridDim, blockDim>>>(
        A.data_ptr<float>(), weight.data_ptr<float>(), bias.data_ptr<float>(), out.data_ptr<float>(), N, K, M);
    cudaDeviceSynchronize();
    return out;
}

// (c) Linear+SiLU layer forward.
torch::Tensor linear_silu_forward(torch::Tensor A, torch::Tensor weight, torch::Tensor bias) {
    TORCH_CHECK(A.is_cuda() && weight.is_cuda() && bias.is_cuda(), "Tensors must be CUDA");
    int N = A.size(0);
    int K = A.size(1);
    int M = weight.size(0);
    auto options = torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCUDA);
    auto out = torch::empty({N, M}, options);
    dim3 blockDim(TILE_DIM, TILE_DIM);
    dim3 gridDim((M + blockDim.x - 1) / blockDim.x, (N + blockDim.y - 1) / blockDim.y);
    linear_silu_kernel<<<gridDim, blockDim>>>(
        A.data_ptr<float>(), weight.data_ptr<float>(), bias.data_ptr<float>(), out.data_ptr<float>(), N, K, M);
    cudaDeviceSynchronize();
    return out;
}

// (d) Full MLP forward: given input t_freq, apply first linear+SiLU then second linear.
torch::Tensor timestep_embed_mlp_forward(torch::Tensor t_freq,
                                         torch::Tensor weight1, torch::Tensor bias1,
                                         torch::Tensor weight2, torch::Tensor bias2) {
    auto hidden = linear_silu_forward(t_freq, weight1, bias1);
    auto out = linear_forward(hidden, weight2, bias2);
    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("timestep_embedding_forward", &timestep_embedding_forward, "Timestep embedding forward (CUDA)");
    m.def("linear_forward", &linear_forward, "Linear forward (CUDA)");
    m.def("linear_silu_forward", &linear_silu_forward, "Linear + SiLU forward (CUDA)");
    m.def("timestep_embed_mlp_forward", &timestep_embed_mlp_forward, "Full MLP forward (CUDA)");
}



EOF


In [8]:
%%bash
cat > cuda_label_embed.cu << 'EOF'
// cuda_label_embed.cu
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

//---------------------------------------------------------
// Label Embedding Kernel
//
// For each sample n and each embedding dimension j:
//   If force_drop[n]==1 then label = num_classes (the extra embedding)
//   Otherwise, label = labels[n].
// Then, out[n, j] = embedding_table[label, j].
//
// labels: [N] (int32)
// force_drop: [N] (int32)  (if not used, pass an array of 0’s)
// embedding_table: [num_classes+1, hidden_size] (float)
// out: [N, hidden_size] (float)
//
__global__ void label_embed_kernel(const int* __restrict__ labels,
                                   const int* __restrict__ force_drop,
                                   const float* __restrict__ embedding_table,
                                   float* __restrict__ out,
                                   int N, int hidden_size, int num_classes) {
    int n = blockIdx.x * blockDim.x + threadIdx.x; // over batch
    int j = blockIdx.y * blockDim.y + threadIdx.y;   // over embedding dim
    if (n < N && j < hidden_size) {
        int label = labels[n];
        if (force_drop[n] == 1)
            label = num_classes;  // Use the extra embedding index
        out[n * hidden_size + j] = embedding_table[label * hidden_size + j];
    }
}

//---------------------------------------------------------
// Wrapper for label embedding.
// If force_drop is not provided, caller can pass a tensor of zeros.
torch::Tensor label_embed_forward(torch::Tensor labels, torch::Tensor embedding_table, torch::Tensor force_drop) {
    TORCH_CHECK(labels.is_cuda(), "labels must be CUDA tensor");
    TORCH_CHECK(embedding_table.is_cuda(), "embedding_table must be CUDA tensor");
    TORCH_CHECK(force_drop.is_cuda(), "force_drop must be CUDA tensor");
    int N = labels.size(0);
    int hidden_size = embedding_table.size(1);
    // num_classes is (num_rows - 1) since the extra row is used for dropout.
    int num_classes = embedding_table.size(0) - 1;
    auto options = torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCUDA);
    auto out = torch::empty({N, hidden_size}, options);
    dim3 blockDim(32, 8);
    dim3 gridDim((N + blockDim.x - 1) / blockDim.x, (hidden_size + blockDim.y - 1) / blockDim.y);
    label_embed_kernel<<<gridDim, blockDim>>>(labels.data_ptr<int>(),
                                              force_drop.data_ptr<int>(),
                                              embedding_table.data_ptr<float>(),
                                              out.data_ptr<float>(),
                                              N, hidden_size, num_classes);
    cudaDeviceSynchronize();
    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("label_embed_forward", &label_embed_forward, "Label Embedding forward (CUDA)");
}

EOF


In [11]:
import math
import torch
import torch.nn as nn

# --- TimestepEmbedder Reference (copied from models.py) ---
class TimestepEmbedderRef(nn.Module):
    def __init__(self, hidden_size, frequency_embedding_size=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size, bias=True),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size, bias=True),
        )
        self.frequency_embedding_size = frequency_embedding_size

    @staticmethod
    def timestep_embedding(t, dim, max_period=10000):
        half = dim // 2
        freqs = torch.exp(
            -math.log(max_period)
            * torch.arange(start=0, end=half, dtype=torch.float32) / half
        ).to(device=t.device)
        args = t[:, None].float() * freqs[None]
        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        if dim % 2:
            embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
        return embedding

    def forward(self, t):
        t_freq = self.timestep_embedding(t, self.frequency_embedding_size)
        t_emb = self.mlp(t_freq)
        return t_emb

# --- Test for TimestepEmbedder ---
N = 32
frequency_embedding_size = 256
hidden_size = 1152
t = torch.rand(N, device='cuda') * 1000  # random timesteps

# Precompute frequencies and move to CUDA.
half = frequency_embedding_size // 2
freqs = torch.exp(-math.log(10000) * torch.arange(0, half, dtype=torch.float32) / half).to('cuda')

# Test sinusoidal embedding kernel.
embed_cuda = timestep_embed.timestep_embedding_forward(t, freqs, frequency_embedding_size)
embed_ref = TimestepEmbedderRef.timestep_embedding(t, frequency_embedding_size)
assert torch.allclose(embed_cuda, embed_ref, atol=1e-5), "Timestep embedding kernel failed"
print("Timestep embedding kernel test passed.")

# Prepare random weights for the MLP layers.
weight1 = torch.randn(hidden_size, frequency_embedding_size, device='cuda')
bias1 = torch.randn(hidden_size, device='cuda')
weight2 = torch.randn(hidden_size, hidden_size, device='cuda')
bias2 = torch.randn(hidden_size, device='cuda')

# Create the reference MLP and move it to CUDA.
ref_mlp = TimestepEmbedderRef(hidden_size, frequency_embedding_size).to('cuda')
with torch.no_grad():
    ref_mlp.mlp[0].weight.copy_(weight1)
    ref_mlp.mlp[0].bias.copy_(bias1)
    ref_mlp.mlp[2].weight.copy_(weight2)
    ref_mlp.mlp[2].bias.copy_(bias2)

out_ref = ref_mlp.forward(t)

# Compute MLP output via the CUDA kernels.
t_freq_cuda = embed_cuda  # our computed sinusoidal embedding
hidden_cuda = timestep_embed.linear_silu_forward(t_freq_cuda, weight1, bias1)
out_cuda = timestep_embed.linear_forward(hidden_cuda, weight2, bias2)

assert torch.allclose(out_cuda, out_ref, atol=1e-3), "Timestep embed MLP kernel failed"
print("Timestep embed MLP kernel test passed.")

# --- Test for LabelEmbedder ---
class LabelEmbedderRef(nn.Module):
    def __init__(self, num_classes, hidden_size, dropout_prob):
        super().__init__()
        use_cfg_embedding = dropout_prob > 0
        self.embedding_table = nn.Embedding(num_classes + use_cfg_embedding, hidden_size)
        self.num_classes = num_classes
        self.dropout_prob = dropout_prob

    def token_drop(self, labels, force_drop_ids=None):
        if force_drop_ids is None:
            drop_ids = torch.rand(labels.shape[0], device=labels.device) < self.dropout_prob
        else:
            drop_ids = force_drop_ids == 1
        labels = torch.where(drop_ids, self.num_classes, labels)
        return labels

    def forward(self, labels, train=True, force_drop_ids=None):
        if (train and self.dropout_prob > 0) or (force_drop_ids is not None):
            labels = self.token_drop(labels, force_drop_ids)
        embeddings = self.embedding_table(labels)
        return embeddings

num_classes = 1000
hidden_size = 1152
dropout_prob = 0.1  # disable dropout for testing

N = 32
labels = torch.randint(0, num_classes, (N,), device='cuda')
force_drop = torch.zeros(N, dtype=torch.int32, device='cuda')
embedding_table = torch.randn(num_classes+1, hidden_size, device='cuda')

embed_ref_module = LabelEmbedderRef(num_classes, hidden_size, dropout_prob).to('cuda')
with torch.no_grad():
    embed_ref_module.embedding_table.weight.copy_(embedding_table)
out_ref = embed_ref_module.forward(labels, train=False, force_drop_ids=torch.zeros_like(labels))
out_cuda = label_embed.label_embed_forward(labels.int(), embedding_table, force_drop)
assert torch.allclose(out_cuda, out_ref, atol=1e-5), "Label embedding kernel failed"
print("Label embedding kernel test passed.")


Timestep embedding kernel test passed.
Timestep embed MLP kernel test passed.
Label embedding kernel test passed.


In [12]:
%%bash
cat > cuda_layernorm.cu << 'EOF'

#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <math.h>

#define BLOCK_SIZE 256
#define WARP_SIZE 32

// Warp-level reduction
__device__ __forceinline__ float warpReduceSum(float val) {
    // Use all 32 threads (full warp)
    for (int offset = WARP_SIZE / 2; offset > 0; offset >>= 1) {
        val += __shfl_down_sync(0xffffffff, val, offset);
    }
    return val;
}

// Block-level reduction using the above warp reduce
__device__ __forceinline__ float blockReduceSum(float val) {
    __shared__ float shared[32];  // one partial sum per warp
    int lane = threadIdx.x % WARP_SIZE;
    int wid  = threadIdx.x / WARP_SIZE;

    // Each warp does a partial reduction
    val = warpReduceSum(val);

    // Write the reduced value of that warp to shared
    if (lane == 0) {
        shared[wid] = val;
    }
    __syncthreads();

    // Now, only the first warp needs to reduce across 'wid' entries
    // The first warp is wid == 0
    val = (threadIdx.x < blockDim.x / WARP_SIZE) ? shared[lane] : 0.0f;
    if (wid == 0) {
        val = warpReduceSum(val);
    }
    // At this point, val is the block-wide sum but
    // only threads in warp0 have the correct value.
    // If you want the final sum to be visible in *every* thread,
    // you need a final broadcast. One approach:

    __shared__ float blockSum;
    if (threadIdx.x == 0) {
        blockSum = val;  // store the final sum in shared[0]
    }
    __syncthreads();

    // Now read it back from shared to "val"
    val = blockSum;
    return val;
}

// -------------------------------------------------------- //
//                   LAYERNORM FORWARD                      //
// -------------------------------------------------------- //
__launch_bounds__(BLOCK_SIZE)
__global__ void layernorm_forward_kernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    float* __restrict__ mean,
    float* __restrict__ rstd,
    const int N,  // batch size
    const int C   // hidden dimension
) {
    int tid = threadIdx.x;
    int bid = blockIdx.x;

    if (bid >= N) return;  // guard

    const float* input_row  = input  + bid * C;
    float* output_row       = output + bid * C;

    // ------------------------------------------------
    // Step 1: compute mean for this row
    // ------------------------------------------------
    float sum = 0.0f;
    for (int i = tid; i < C; i += BLOCK_SIZE) {
        sum += input_row[i];
    }
    sum = blockReduceSum(sum);
    float mu = sum / C;

    // We want all threads in the block to see the same mu
    __shared__ float mu_shared;
    if (tid == 0) {
        mean[bid] = mu;    // optional: store the mean
        mu_shared = mu;    // broadcast via shared memory
    }
    __syncthreads();
    mu = mu_shared;        // now every thread reads mu

    // ------------------------------------------------
    // Step 2: compute variance and rstd
    // ------------------------------------------------
    float var_sum = 0.0f;
    for (int i = tid; i < C; i += BLOCK_SIZE) {
        float diff = input_row[i] - mu;
        var_sum   += diff * diff;
    }
    var_sum = blockReduceSum(var_sum);
    float sigma     = sqrtf(var_sum / C + 1e-5f);
    float inv_sigma = 1.0f / sigma;

    // Again, broadcast inv_sigma so all threads see it
    __shared__ float inv_sigma_shared;
    if (tid == 0) {
        rstd[bid] = inv_sigma;
        inv_sigma_shared = inv_sigma;
    }
    __syncthreads();
    inv_sigma = inv_sigma_shared;

    // ------------------------------------------------
    // Step 3: normalize
    // ------------------------------------------------
    for (int i = tid; i < C; i += BLOCK_SIZE) {
        output_row[i] = (input_row[i] - mu) * inv_sigma;
    }
}

torch::Tensor layernorm_forward(torch::Tensor input) {
    TORCH_CHECK(input.is_cuda(), "Input must be a CUDA tensor");
    TORCH_CHECK(input.dim() == 2, "Input must be 2D (batch_size, hidden_size)");
    TORCH_CHECK(input.scalar_type() == torch::ScalarType::Float,
        "Input must be float32");

    const int N = input.size(0);
    const int C = input.size(1);

    auto options = torch::TensorOptions()
        .dtype(torch::kFloat32)
        .device(input.device());

    auto output = torch::empty_like(input);
    auto mean   = torch::empty({N}, options);
    auto rstd   = torch::empty({N}, options);

    const dim3 blocks(N);
    const dim3 threads(BLOCK_SIZE);

    layernorm_forward_kernel<<<blocks, threads>>>(
        input.data_ptr<float>(),
        output.data_ptr<float>(),
        mean.data_ptr<float>(),
        rstd.data_ptr<float>(),
        N, C
    );
    return output;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("layernorm_forward", &layernorm_forward, "LayerNorm forward (CUDA)");
}

EOF


In [13]:
import torch
from torch.utils.cpp_extension import load
import os


# Compile the extension with optimizations for A100
cuda_layernorm = load(
    name="cuda_layernorm",
    sources=["cuda_layernorm.cu"],
    verbose=True,
    extra_cuda_cflags=[
        '-O3',
        '--use_fast_math',
        '-lineinfo'
    ]
)

# Test function
def test_layernorm():
    B, C = 128, 768
    x = torch.randn(B, C, device='cuda', dtype=torch.float32)

    # Run our custom implementation
    out_cuda = cuda_layernorm.layernorm_forward(x)

    # Run PyTorch's implementation as reference
    layer_norm = torch.nn.LayerNorm(C, elementwise_affine=False).cuda()
    out_ref = layer_norm(x)

    # Compare results
    max_error = (out_cuda - out_ref).abs().max().item()
    print(f"Max absolute error: {max_error}")
    assert torch.allclose(out_cuda, out_ref, atol=1e-5)
    print("LayerNorm test passed!")

test_layernorm()

Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/cuda_layernorm/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:1964: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module cuda_layernorm...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


Max absolute error: 4.76837158203125e-07
LayerNorm test passed!


Loading extension module cuda_layernorm...


In [28]:
%%bash
cat > cuda_attention.cu << 'EOF'
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <math.h>

#define THREADS 256

// A very simple forward attention kernel.
// Each block computes one output row: out[b, i, :] = sum_j softmax(scores)[j] * V[b, j, :],
// where scores[j] = (Q[b, i, :] dot K[b, j, :]) / sqrt(C).
// For simplicity, only thread 0 of each block computes the weighted sum.
__global__ void attention_forward_kernel(const float* __restrict__ Q,
                                           const float* __restrict__ K,
                                           const float* __restrict__ V,
                                           float* __restrict__ out,
                                           int B, int T, int C) {
    // Grid: (B, T)
    int b = blockIdx.x;  // batch index
    int i = blockIdx.y;  // query (sequence) index
    int tid = threadIdx.x;

    // Shared memory: we'll use T floats to store the dot-product scores.
    extern __shared__ float sdata[];  // size: T * sizeof(float)
    float* scores = sdata; // scores[0..T-1]

    // --- Compute dot-product scores for query Q[b,i,:] with each key K[b,j,:] ---
    // Each thread loops over j in [tid, T, blockDim.x].
    for (int j = tid; j < T; j += blockDim.x) {
        float dot = 0.0f;
        int q_offset = b * T * C + i * C;       // Q[b,i,:]
        int k_offset = b * T * C + j * C;         // K[b,j,:]
        for (int k = 0; k < C; k++) {
            dot += Q[q_offset + k] * K[k_offset + k];
        }
        // Scale by 1/sqrt(C)
        scores[j] = dot / sqrtf((float)C);
    }
    __syncthreads();

    // --- Compute maximum value in scores for numerical stability ---
    float max_val = -1e20f;
    for (int j = tid; j < T; j += blockDim.x) {
        if (scores[j] > max_val)
            max_val = scores[j];
    }
    // Use shared memory for reduction (assume blockDim.x <= THREADS)
    __shared__ float smax[THREADS];
    smax[tid] = max_val;
    __syncthreads();
    for (int stride = blockDim.x / 2; stride > 0; stride /= 2) {
        if (tid < stride) {
            if (smax[tid + stride] > smax[tid])
                smax[tid] = smax[tid + stride];
        }
        __syncthreads();
    }
    max_val = smax[0];

    // --- Compute exponentials and their sum ---
    float sum = 0.0f;
    for (int j = tid; j < T; j += blockDim.x) {
        float exp_val = expf(scores[j] - max_val);
        scores[j] = exp_val;  // store the exponentiated value back
        sum += exp_val;
    }
    __syncthreads();
    __shared__ float ssum[THREADS];
    ssum[tid] = sum;
    __syncthreads();
    for (int stride = blockDim.x / 2; stride > 0; stride /= 2) {
        if (tid < stride) {
            ssum[tid] += ssum[tid + stride];
        }
        __syncthreads();
    }
    sum = ssum[0];

    // --- Normalize scores to form the softmax ---
    for (int j = tid; j < T; j += blockDim.x) {
        scores[j] /= sum;
    }
    __syncthreads();

    // --- Compute weighted sum over V to form the output ---
    // For simplicity, we let thread 0 do the accumulation.
    if (tid == 0) {
        for (int k = 0; k < C; k++) {
            float acc = 0.0f;
            for (int j = 0; j < T; j++) {
                int v_offset = b * T * C + j * C;
                acc += scores[j] * V[v_offset + k];
            }
            out[b * T * C + i * C + k] = acc;
        }
    }
}

// C++ interface for the forward attention function.
torch::Tensor attention_forward(torch::Tensor Q,
                                torch::Tensor K,
                                torch::Tensor V) {
    TORCH_CHECK(Q.is_cuda(), "Q must be a CUDA tensor");
    TORCH_CHECK(K.is_cuda(), "K must be a CUDA tensor");
    TORCH_CHECK(V.is_cuda(), "V must be a CUDA tensor");
    TORCH_CHECK(Q.dtype() == torch::kFloat32, "Only float32 supported");
    TORCH_CHECK(K.dtype() == torch::kFloat32, "Only float32 supported");
    TORCH_CHECK(V.dtype() == torch::kFloat32, "Only float32 supported");

    int B = Q.size(0);
    int T = Q.size(1);
    int C = Q.size(2);

    auto out = torch::empty({B, T, C}, Q.options());

    // Launch one block per (batch, sequence) pair.
    // Grid dimensions: (B, T), blockDim.x = THREADS.
    // Shared memory size: T * sizeof(float) (for the scores).
    dim3 grid(B, T);
    dim3 block(THREADS);
    size_t shared_memory_size = T * sizeof(float);
    attention_forward_kernel<<<grid, block, shared_memory_size>>>(
        Q.data_ptr<float>(),
        K.data_ptr<float>(),
        V.data_ptr<float>(),
        out.data_ptr<float>(),
        B, T, C
    );
    cudaDeviceSynchronize();
    return out;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("attention_forward", &attention_forward, "Attention Forward (CUDA)");
}
EOF


In [32]:
import torch
import torch.nn.functional as F
from torch.utils.cpp_extension import load
import time
import numpy as np

cuda_attention = load(
    name="cuda_attention",
    sources=["cuda_attention.cu"],
    extra_cuda_cflags=["-O3"],
    verbose=True
)

def check_correctness(Q, K, V):
    """Verify CUDA implementation matches PyTorch reference"""
    print("\nVerifying Implementation Correctness")
    print("-" * 50)

    # Get outputs from both implementations
    out_cuda = cuda_attention.attention_forward(Q, K, V)
    out_ref = torch.matmul(
        F.softmax(torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(Q.size(-1)), dim=-1),
        V
    )

    # Check error metrics
    max_error = (out_cuda - out_ref).abs().max().item()
    mean_error = (out_cuda - out_ref).abs().mean().item()

    print(f"Max absolute error:  {max_error:.6f}")
    print(f"Mean absolute error: {mean_error:.6f}")

    if torch.allclose(out_cuda, out_ref, atol=1e-5):
        print("✓ Implementation matches reference")
        return True
    else:
        print("✗ Implementation does not match reference!")
        return False

def benchmark_attention(batch_size=1, seq_len=32, dim=64, trials=5, warmup=100, iters=1000):
    """Benchmark attention implementation focusing on optimal configuration"""
    print("\nAttention Performance Analysis")
    print("=" * 50)
    print(f"Configuration: {batch_size}x{seq_len}x{dim}")

    # Create tensors
    torch.manual_seed(42)  # For reproducibility
    Q = torch.randn(batch_size, seq_len, dim, device='cuda', dtype=torch.float32)
    K = torch.randn(batch_size, seq_len, dim, device='cuda', dtype=torch.float32)
    V = torch.randn(batch_size, seq_len, dim, device='cuda', dtype=torch.float32)

    # First verify correctness
    if not check_correctness(Q, K, V):
        print("\nSkipping benchmark due to implementation mismatch!")
        return

    # Extensive warmup
    print(f"\nWarming up ({warmup} iterations)...")
    for _ in range(warmup):
        _ = cuda_attention.attention_forward(Q, K, V)
        _ = torch.matmul(F.softmax(torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(dim), dim=-1), V)
    torch.cuda.synchronize()

    # Multiple trials
    cuda_times = []
    torch_times = []

    print(f"Running {trials} trials, {iters} iterations each...")
    for trial in range(trials):
        # CUDA implementation timing
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(iters):
            _ = cuda_attention.attention_forward(Q, K, V)
        torch.cuda.synchronize()
        cuda_time = (time.perf_counter() - start) / iters * 1000  # ms
        cuda_times.append(cuda_time)

        # PyTorch reference timing
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(iters):
            _ = torch.matmul(F.softmax(torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(dim), dim=-1), V)
        torch.cuda.synchronize()
        torch_time = (time.perf_counter() - start) / iters * 1000  # ms
        torch_times.append(torch_time)

    # Get best results (lowest latency)
    best_cuda = min(cuda_times)
    best_torch = min(torch_times)

    # Calculate metrics
    speedup = best_torch / best_cuda
    improvement = (best_torch - best_cuda) / best_torch * 100
    tokens_per_sec_cuda = batch_size * seq_len / (best_cuda / 1000)
    tokens_per_sec_torch = batch_size * seq_len / (best_torch / 1000)

    print("\nPerformance Results")
    print("-" * 50)
    print(f"Best Latency (over {trials} trials):")
    print(f"  CUDA Implementation:  {best_cuda:.3f} ms")
    print(f"  PyTorch Reference:    {best_torch:.3f} ms")
    print(f"  Speedup:             {speedup:.2f}x")
    print(f"  Performance Gain:    {improvement:.1f}%")

    print(f"\nThroughput:")
    print(f"  CUDA Implementation:  {tokens_per_sec_cuda/1000:.1f}k tokens/sec")
    print(f"  PyTorch Reference:    {tokens_per_sec_torch/1000:.1f}k tokens/sec")
    print(f"  Throughput Ratio:    {tokens_per_sec_cuda/tokens_per_sec_torch:.2f}x")

if __name__ == "__main__":
    # Run with optimal configuration
    benchmark_attention()

Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
No modifications detected for re-loaded extension module cuda_attention, skipping build step...
Loading extension module cuda_attention...



Attention Performance Analysis
Configuration: 1x32x64

Verifying Implementation Correctness
--------------------------------------------------
Max absolute error:  0.000000
Mean absolute error: 0.000000
✓ Implementation matches reference

Warming up (100 iterations)...
Running 5 trials, 1000 iterations each...

Performance Results
--------------------------------------------------
Best Latency (over 5 trials):
  CUDA Implementation:  0.058 ms
  PyTorch Reference:    0.096 ms
  Speedup:             1.66x
  Performance Gain:    39.6%

Throughput:
  CUDA Implementation:  550.8k tokens/sec
  PyTorch Reference:    332.6k tokens/sec
  Throughput Ratio:    1.66x
